# NeedleTTT — shared Drive + GitHub setup (DEE-30)

This notebook is an **environment smoke test**, not a training run. It mounts your Google Drive, verifies the shared **NeedleTTT** shortcut/folder and directory layout, clones the GitHub repository, and imports the `needlettt` package.

**Before running:** The folder owner must share one actual Drive folder with all teammates (Editor access). Each teammate should add a shortcut named `NeedleTTT` at the top of **My Drive**. Follow [docs/colab_workflow.md](https://github.com/muyishen2040/NeedleTTT/blob/main/docs/colab_workflow.md) for directory structure and access checks.

**Security:** This notebook does not create/share the Google Drive folder, request login tokens for GitHub, download real datasets, or run CUDA. Do not paste private credentials or course Needle solutions here.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## Check the shared workspace

Default path: `/content/drive/MyDrive/NeedleTTT` (the shortcut points to the one team-shared folder). If you use a different shortcut location, set `os.environ["NEEDLETTT_DRIVE_ROOT"]` before running this cell. Do **not** create a new folder just to make a missing path pass.

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path(
    os.environ.get("NEEDLETTT_DRIVE_ROOT", "/content/drive/MyDrive/NeedleTTT")
).expanduser()
DATA_ROOT = DRIVE_ROOT / "data"
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
RESULTS_ROOT = DRIVE_ROOT / "results"

required_dirs = [
    DRIVE_ROOT,
    DATA_ROOT / "raw",
    DATA_ROOT / "processed",
    CHECKPOINT_ROOT,
    RESULTS_ROOT,
]
missing = [str(path) for path in required_dirs if not path.is_dir()]

if missing:
    raise FileNotFoundError(
        "Shared NeedleTTT Drive paths missing:\n- "
        + "\n- ".join(missing)
        + "\nAsk the folder owner to create/share these paths and add "
          "a My Drive shortcut named NeedleTTT. Do not create a separate copy."
    )

os.environ["DATA_ROOT"] = str(DATA_ROOT)
print("Shared folder path:", DRIVE_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("Directory structure: OK (sharing permissions must be tested separately)")

## Clone GitHub code and install package

The source code and notebooks live in GitHub, **not** in Drive. A Colab runtime is ephemeral, so a clone under `/content` is temporary. This checks out `main` once; if you already have the repository in this runtime, it keeps your existing checkout without silently changing the revision.

Currently the repository is public. If your team makes it private, use an approved authenticated clone mechanism (e.g. Colab secrets) **without** putting access tokens into this notebook.

In [ ]:
import subprocess
import sys

REPO_URL = "https://github.com/muyishen2040/NeedleTTT.git"
REPO_DIR = Path("/content/NeedleTTT")

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
    print("Using existing checkout:", REPO_DIR)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)
GIT_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("Git commit:", GIT_COMMIT)

In [ ]:
import needlettt
from needlettt.contracts import ItemCatalog

demo_catalog = ItemCatalog.from_json(REPO_DIR / "data" / "examples" / "catalog.json")
print("NeedleTTT import: OK")
print("Fictional demo catalog items:", len(demo_catalog.items))
print("This is only a scaffold smoke test, NOT a trained model or real dataset.")
print("Ready to read processed data from:", DATA_ROOT / "processed")

## Next steps

- Ask **Member A / DEE-6** which `data/processed/<dataset>/<version>/` to use. Record its `manifest.json`, split seed and item-ID map.
- Put model artifacts under `checkpoints/<model>/<run_id>/` and metrics under `results/<run_id>/` (unique run IDs prevent overwrites).
- Each teammate runs this notebook in their **own account** and confirms access to the *same* shared test file. A passing path check alone does not prove the folder is shared.
- Report any setup issue on [Linear DEE-30](https://linear.app/deep-learning-system-final/issue/DEE-30/set-up-shared-google-drive-and-colab-workspace).